In [88]:
!pip install faker -q

In [89]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [90]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA = "/content/data"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_bersih.csv', 'transaksi_mentah.csv']


In [91]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)
N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "
    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [92]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [93]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah drop_duplicates():", len(df))

Jumlah baris setelah drop_duplicates(): 495


In [94]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction id duplicate:", df['transaction_id'].duplicated().sum())
df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


In [95]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

In [96]:
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [97]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [98]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [99]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [100]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


In [101]:
df["is_valid_price"] = df["price"] > 0
print("Jumlah harga tidak valid (<= 0 atau NaN):", (~df["is_valid_price"]).sum())

Jumlah harga tidak valid (<= 0 atau NaN): 0


In [102]:
print("\nJumlah transaksi per kategori:")
print(df["category"].value_counts())


Jumlah transaksi per kategori:
category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64


**Jawaban Studi Kasus**
1. Perbedaan jumlah ini wajar terjadi karena laporan 515 baris dari tim IT masih berupa data mentah yang belum melalui tahap pembersihan. Setelah dilakukan proses preprocessing, ada 25 baris yang harus dibuang sehingga tersisa 490 baris. Rinciannya: 15 baris dihapus karena terdeteksi sebagai data duplikat (transaksi yang sama tercatat dua kali), dan 10 baris lainnya dihapus karena kehilangan informasi wajib, yaitu nama pelanggan dan metode pembayaran.

2. data 490 baris tersebut jauh lebih valid untuk digunakan. Dalam Veracity, yaitu seberapa akurat dan dapat dipercayanya suatu data. Percuma memiliki Volume data yang lebih besar (515 baris) jika di dalamnya terdapat duplikasi atau data yang bolong. Jika tim Finance tetap memakai 515 baris, perhitungan pendapatan mereka pasti salah karena ada transaksi fiktif/ganda yang ikut terhitung.

3. Kolom rating sengaja dibiarkan kosong karena dari awal pengisian rating di sistem marketplace memang bersifat opsional bagi pembeli. Jika kita mengisi kekosongan tersebut dengan angka tebakan hal itu justru akan memanipulasi data. Untuk menghitung "rating rata-rata semua transaksi", tim Finance cukup menghitung rata-rata dari data yang memang diberikan rating oleh pembelinya saja, sementara data yang kosong dibiarkan saja dari perhitungan statistik tersebut.